In [1]:
import asyncio
import os
import time
from typing import Literal, Optional
from dotenv import load_dotenv

from openai import AsyncOpenAI
import pandas as pd
from pydantic import BaseModel, Field
from tqdm.notebook import tqdm

from pprint import pprint

load_dotenv()

True

In [2]:
client = AsyncOpenAI(
    base_url=os.getenv("GENERATION_API_URL"),
    #base_url="https://f3948611-98f8-4738-b973-30b4a657cffa.ifr.fr-par.scaleway.com/v1",
    api_key=os.getenv("SCW_SECRET_KEY"),
)

In [34]:
#df = pd.read_parquet('data/conclusions_sample_10k_2026-01-27.parquet')
df = pd.read_parquet('results_557k/sample_2000_extracted_policies.parquet')

In [35]:
tax = pd.read_parquet('results_557k/sample_2000_taxonomy_260303.parquet')

In [36]:
df = df.merge(tax.drop(columns='text'), on=['openalex_id', 'chunk_idx'], how='left')

In [51]:
pdf = df[df.contains_policies==True].explode('policies').reset_index(drop=True)
pdf = pdf[pdf.policies.notna()].reset_index(drop=True)

In [162]:
impact_cols = ['justice_consideration_pred', 'planetary_boundaries_pred', 'wellbeing_pred', 'human_needs_pred', 'natural_ressource_pred']

system_prompt = """
Your job is to precisely extract policy information from scientific texts, more specifically the impact direction of a policy along given impact dimensions.
Direction can be:
- positive if the text clearly suggests a beneficial impact of the policy on the dimension
- negative if the text clearly suggests a detrimental impact of the policy on the dimension
- neutral if the text suggests no significant or mixed impact of the policy on the dimension
- unknown if either the impact dimension is irrelevant to the text or if its direction cannot be inferred.

For planetary boundaries, positive means that the policy helps to stay within the boundaries (reducing climate change is positive), while negative means that it contributes to exceeding them.
For natural ressources, positive means that the policy helps to preserve them (reducing consumption), while negative means that it contributes to their depletion.
"""

def build_prompt(row):
    prompt = f"""Here is an excerpt from a scientific article: 
{row.text}

Here is a public policy extracted from this article: 
{row.policies.split(']')[-1].strip()}

Extract the direction of impact along the following dimensions:
"""

    labels = []
    for col in impact_cols:
        if len(row[col]) > 0:
            prompt += f"- {col.replace('_pred', '').replace('_', ' ').title()}:\n"
            for label in row[col]:
                labels += [label]
                prompt += f"  - {label}\n\n"

    prompt += "YOU MUST refer the the second-level labels in your answer: " + ", ".join(labels)

    return prompt, labels


In [129]:
print(build_prompt(pdf.iloc[0]))

('Here is an excerpt from a scientific article: \n considered relevant in the very early phases of deployment, other measures that encompass flexibility should be applied instead.\n5.5 Regulation: Dissecting an umbrella-category\nThroughout the reviewed literature, regulation is a recurring category [29,37,39,44–49,51,52,54,57,58]. In this study it proved relevant to subdivide regulation into separate issues to increase detail in the taxonomy.\n5.6 Common solutions\nMany of the barriers identified can be perceived as policy and regulatory risks. Especially barriers related to investment, but also bounded rationality and acceptance. Long-term policies and targets can mitigate those risks. In addition to the solutions in 3.2, is research and development [7,29,86]. Another option is direct\nPage 22 of 31 regulation applying command and control solutions by government fiat. E.g. in the case of Denmark, where\nDE must prove a >1 500 DKK/year/consumer saving, to be allowed to choose a biomas

In [138]:
model_name = "mistral-small-3.2-24b-instruct-2506"
#model_name = "qwen/qwen3-235b-a22b-instruct-2507"

In [169]:
async def extract_impacts(
    row, model_name: str, client: AsyncOpenAI = client
):
    try:
        prompt, labels = build_prompt(row)
        class ImpactExtractionResponse(BaseModel):
            impacts: list[tuple[Literal[*labels], Literal['positive', 'negative', 'neutral', 'unknown']]] = Field(..., description="List of tuples (dimension, direction)")

        response = await client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt.strip()},
                {"role": "user", "content": prompt.strip()},
            ],
            temperature=0,
            max_tokens=512,
            response_format= {
                    "type": "json_schema",
                    "json_schema": {
                        "name": ImpactExtractionResponse.__name__,
                        "schema": ImpactExtractionResponse.model_json_schema(),
                    },
                },
            timeout=60,
            stream=False,
        )
        return ImpactExtractionResponse.model_validate_json(response.choices[0].message.content)
    except Exception as e:
        print('Error:', e)
        return f"Error: {e}"


async def batch_extract_impacts(
    batch: pd.DataFrame, model_name: str, client: AsyncOpenAI = client
):
    results = await asyncio.gather(*[extract_impacts(row, model_name, client) for _, row in batch.iterrows()])
    return results

In [156]:
res = await extract_impacts(pdf.iloc[173], model_name, client)
res

ImpactExtractionResponse(impacts=[('distributional', 'neutral'), ('land_system_change', 'neutral'), ('environment', 'neutral'), ('shelter_and_living_conditions', 'neutral'), ('urban_land', 'neutral')])

In [164]:
res = await batch_extract_impacts(pdf.iloc[20:30], model_name, client)
res

[ImpactExtractionResponse(impacts=[('distributional', 'unknown'), ('climate_change', 'positive'), ('environment', 'positive')]),
 ImpactExtractionResponse(impacts=[('distributional', 'unknown'), ('climate_change', 'positive'), ('environment', 'positive')]),
 ImpactExtractionResponse(impacts=[('distributional', 'unknown'), ('climate_change', 'positive'), ('environment', 'positive')]),
 ImpactExtractionResponse(impacts=[('environment', 'unknown'), ('jobs', 'unknown')]),
 ImpactExtractionResponse(impacts=[('environment', 'unknown'), ('jobs', 'unknown')]),
 ImpactExtractionResponse(impacts=[('environment', 'unknown'), ('jobs', 'unknown')]),
 ImpactExtractionResponse(impacts=[('environment', 'unknown'), ('jobs', 'unknown')]),
 ImpactExtractionResponse(impacts=[('environment', 'neutral'), ('jobs', 'neutral')]),
 ImpactExtractionResponse(impacts=[('environment', 'unknown'), ('jobs', 'unknown')]),
 ImpactExtractionResponse(impacts=[('environment', 'unknown'), ('jobs', 'unknown')])]

In [171]:
# process by batch
batch_size = 20
rpm_quota = 600
wait_time = 60 / (rpm_quota / batch_size)
results = []
for i in tqdm(range(0, len(pdf), batch_size)):
    batch = pdf.iloc[i:i+batch_size]
    preds = await batch_extract_impacts(batch, model_name, client)
    results.extend(preds)
    time.sleep(0.1)

  0%|          | 0/119 [00:00<?, ?it/s]

Error: 1 validation error for ImpactExtractionResponse
  Invalid JSON: EOF while parsing a list at line 2 column 507 [type=json_invalid, input_value='{ "impacts": [\n   \t\t\...t\t\t\t\t\t\t\t\t\t\t\t', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid
Error: 1 validation error for ImpactExtractionResponse
  Invalid JSON: EOF while parsing a list at line 2 column 507 [type=json_invalid, input_value='{ "impacts": [\n   \t\t\...t\t\t\t\t\t\t\t\t\t\t\t', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid
Error: 1 validation error for ImpactExtractionResponse
  Invalid JSON: EOF while parsing a list at line 2 column 507 [type=json_invalid, input_value='{ "impacts": [\n   \t\t\...t\t\t\t\t\t\t\t\t\t\t\t', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid
Error: 1 validation error for ImpactExtractionResponse
  Invalid JSON: EOF while parsing a list a

In [201]:
s = 0
for r in results:
    if isinstance(r, str) and r.startswith("Error:"):
        s += 1
print(f"{s} errors out of {len(results)} samples")

97 errors out of 2377 samples


In [194]:
pdf['impacts_directions'] = [r.impacts if not isinstance(r, str) else [("error", "error")] for r in results]

In [195]:
pdf.to_parquet('results_557k/sample_2000_policies_impact_directions_2026-03-04.parquet')